In [118]:
import os
import geopandas as gpd
import pandas as pd

#Get current working directory
project_folder = os.getcwd()

#Build full path to the GeoPackage
farm_file_path = "C:/Users/Vasmai/dataset/farm_boundaries_sampled.gpkg"

#Load the spatial farm boundary dataset
farm_boundaries = gpd.read_file(farm_file_path)

#get first few records
print(farm_boundaries.head())

#Display column names
print(farm_boundaries.columns)

#Export the full dataset to CSV
csv_file_path = os.path.join("C:/Users/Vasmai/dataset", "farm_boundaries.csv")

farm_boundaries.to_csv(csv_file_path, index=False)

print(f"\nGeoPackage exported to CSV: {csv_file_path}")
# Check dataset structure, data types, and missing values
farm_boundaries.info()

# Dataset not included in GitHub due to privacy
# Can be downloaded from MS Teams → Planting Optimisation Tool → Datasets → GIS

                name treeo2_id  district subdistrict         suku  \
0     Abel Pereira_1              BAUCAU      BAGUIA     Lavateri   
1     Abel Pereira_2              BAUCAU      BAGUIA     Lavateri   
2    Agostinho Alves            VIQUEQUE  UATUCARBAU  Alaua-Craik   
3  Agostinho Freitas            VIQUEQUE  UATUCARBAU  Alaua-Craik   
4     Amelia Menezes              BAUCAU      BAGUIA  Alaua-Craik   

         aldeia     texture   ph  avg_annual_rainfall_2020-2024  \
0  Onortibalari        Clay  6.2                           1959   
1  Onortibalari        Clay  6.2                           1959   
2      Neolidae  Sandy Loam  8.2                           2021   
3      Neolidae  Sandy Loam  5.9                           2554   
4      Neolidae        Clay  7.0                           2021   

   avg_annual_temperature_2020-2024  elevation          area  area_ha  \
0                              24.0        585   3596.997671     0.36   
1                              24.0 

#### Remove Privacy Information

In [119]:
#getrid of privacy info
cols_to_remove = ['name', 'suku', 'aldeia', 'treeo2_id', 'district', 'subdistrict', 'area']
farm_cleaned = farm_boundaries.drop(columns=cols_to_remove, errors='ignore')

# Preview cleaned dataset
print("privacy info removed:")

# Check cleaned dataset info
farm_cleaned.info()

farm_cleaned.head()

privacy info removed:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 3201 entries, 0 to 3200
Data columns (total 7 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   texture                           3154 non-null   object  
 1   ph                                3154 non-null   object  
 2   avg_annual_rainfall_2020-2024     3201 non-null   int32   
 3   avg_annual_temperature_2020-2024  3193 non-null   float64 
 4   elevation                         3201 non-null   int32   
 5   area_ha                           3201 non-null   float64 
 6   geometry                          3201 non-null   geometry
dtypes: float64(2), geometry(1), int32(2), object(2)
memory usage: 150.2+ KB


,texture,ph,avg_annual_rainfall_2020-2024,avg_annual_temperature_2020-2024,elevation,area_ha,geometry
0,Clay,6.2,1959,24.0,585,0.36,"MULTIPOLYGON Z (((904783.597 9050919.352 0, 90..."
1,Clay,6.2,1959,24.0,481,0.48,"MULTIPOLYGON Z (((905235.474 9051056.651 0, 90..."
2,Sandy Loam,8.2,2021,25.0,179,1.18,"MULTIPOLYGON Z (((903414.205 9042747.721 0, 90..."
3,Sandy Loam,5.9,2554,24.0,260,0.46,"MULTIPOLYGON Z (((902007.836 9043097.668 0, 90..."
4,Clay,7.0,2021,25.0,129,2.00,"MULTIPOLYGON Z (((904016.568 9042595.439 0, 90..."


#### Rename Columns

In [131]:
# add farm_id and rename columns
farm_cleaned.insert(0, 'farm_id', range(1, len(farm_boundaries) + 1))
farm_cleaned = farm_cleaned.rename(columns={'avg_annual_rainfall_2020-2024': 'rainfall','avg_annual_temperature_2020-2024': 'temperature'})
farm_cleaned.info()
farm_cleaned.head()

ValueError: cannot insert farm_id, already exists

#### Missing Values

In [121]:
farm_cleaned.isnull().sum()

farm_id         0
texture        47
ph             47
rainfall        0
temperature     8
elevation       0
area_ha         0
geometry        0
dtype: int64

In [122]:
#Convert 'ph' column to float and round to 1 decimal
farm_cleaned['ph'] = farm_cleaned['ph'].astype(float).round(1)

# Fill missing pH values
farm_cleaned['ph'] = farm_cleaned['ph'].fillna(farm_cleaned['ph'].median())

# Fill missing temperature values
farm_cleaned['temperature'] = farm_cleaned['temperature'].fillna(farm_cleaned['temperature'].median())

# Fill missing texture values
farm_cleaned['texture'] = farm_cleaned['texture'].fillna(farm_cleaned['texture'].mode()[0])

# Check missing values again
print(farm_cleaned.isnull().sum())
print(farm_cleaned.head())

farm_id        0
texture        0
ph             0
rainfall       0
temperature    0
elevation      0
area_ha        0
geometry       0
dtype: int64
   farm_id     texture   ph  rainfall  temperature  elevation  area_ha  \
0        1        Clay  6.2      1959         24.0        585     0.36   
1        2        Clay  6.2      1959         24.0        481     0.48   
2        3  Sandy Loam  8.2      2021         25.0        179     1.18   
3        4  Sandy Loam  5.9      2554         24.0        260     0.46   
4        5        Clay  7.0      2021         25.0        129     2.00   

                                            geometry  
0  MULTIPOLYGON Z (((904783.597 9050919.352 0, 90...  
1  MULTIPOLYGON Z (((905235.474 9051056.651 0, 90...  
2  MULTIPOLYGON Z (((903414.205 9042747.721 0, 90...  
3  MULTIPOLYGON Z (((902007.836 9043097.668 0, 90...  
4  MULTIPOLYGON Z (((904016.568 9042595.439 0, 90...  


#### Data Validation after cleaning data

#### Check datatypes

In [123]:
print(farm_cleaned.dtypes)

farm_id           int64
texture          object
ph              float64
rainfall          int32
temperature     float64
elevation         int32
area_ha         float64
geometry       geometry
dtype: object


#### Missing Values

In [124]:
print(farm_cleaned.isnull().sum())

farm_id        0
texture        0
ph             0
rainfall       0
temperature    0
elevation      0
area_ha        0
geometry       0
dtype: int64


#### Valid Range Check

In [125]:
# Describe
print(farm_cleaned.describe())

           farm_id           ph     rainfall  temperature    elevation  \
count  3201.000000  3201.000000  3201.000000  3201.000000  3201.000000   
mean   1601.000000     6.983099  1807.820056    24.193689   517.453296   
std     924.193432     0.879552   347.553724     1.457514   254.959382   
min       1.000000     5.600000   865.000000    19.000000     9.000000   
25%     801.000000     6.200000  1529.000000    23.000000   353.000000   
50%    1601.000000     6.900000  1734.000000    24.000000   490.000000   
75%    2401.000000     7.900000  2034.000000    25.000000   711.000000   
max    3201.000000     8.500000  2624.000000    28.000000  1313.000000   

           area_ha  
count  3201.000000  
mean      1.413046  
std       2.973851  
min       0.000000  
25%       0.390000  
50%       0.750000  
75%       1.410000  
max      85.760000  


#### Replaced "area_ha" with median

In [126]:
median_area = farm_cleaned['area_ha'].median()
farm_cleaned['area_ha'] = farm_cleaned['area_ha'].replace(0, median_area)
print(median_area)

print((farm_cleaned['area_ha'] == 0).sum())

0.75
0


In [134]:
print(farm_cleaned.describe())

           farm_id           ph     rainfall  temperature    elevation  \
count  3201.000000  3201.000000  3201.000000  3201.000000  3201.000000   
mean   1601.000000     6.983099  1807.820056    24.193689   517.453296   
std     924.193432     0.879552   347.553724     1.457514   254.959382   
min       1.000000     5.600000   865.000000    19.000000     9.000000   
25%     801.000000     6.200000  1529.000000    23.000000   353.000000   
50%    1601.000000     6.900000  1734.000000    24.000000   490.000000   
75%    2401.000000     7.900000  2034.000000    25.000000   711.000000   
max    3201.000000     8.500000  2624.000000    28.000000  1313.000000   

           area_ha  
count  3201.000000  
mean      1.413749  
std       2.973606  
min       0.010000  
25%       0.390000  
50%       0.750000  
75%       1.410000  
max      85.760000  


#### Categorical validation

In [128]:
print(farm_cleaned['texture'].unique())

['Clay' 'Sandy Loam' 'Clay Loam' 'Loam' 'Variable' 'Silty Loam'
 'Sandy Clay' 'Organic' 'Silty Clay']


### Spatial Consistency Check

#### Check missing geometries

In [140]:
missing_farm_boundaries = farm_boundaries.geometry.isnull().sum()
missing_farm_cleaned = farm_cleaned.geometry.isnull().sum()
print(missing_farm_boundaries)
print(missing_farm_cleaned)

0
0


#### Check for duplicate farm_id

In [141]:
# Check duplicates in farm_cleaned
print(farm_cleaned['farm_id'].duplicated().sum())

# Check duplicates in farm_boundaries
print(farm_boundaries['farm_id'].duplicated().sum())

0
0


#### Check Geometry validity

In [136]:
print(farm_boundaries.geometry.is_valid.value_counts())

True    3201
Name: count, dtype: int64


#### One-to-One 

In [137]:
num_cleaned_records = len(farm_cleaned)
num_boundary_records = len(farm_boundaries)
one_to_one_issue = num_cleaned_records != num_cleaned_records
print("One-to-one relationship :", one_to_one_issue, "\n")

One-to-one relationship : False 



#### Area Mismatch

In [138]:
farm_boundaries = farm_boundaries.reset_index(drop=True)
farm_boundaries['farm_id'] = range(1, len(farm_boundaries)+1)

farm_boundaries = farm_boundaries.to_crs(epsg=3857)  # meters for accurate area
farm_boundaries['calc_area_ha'] = farm_boundaries.geometry.area / 10000  # hectares

# Merge with master table
check_area = farm_cleaned.merge(farm_boundaries[['farm_id','calc_area_ha']], on='farm_id')
check_area['abs_diff'] = (check_area['calc_area_ha'] - check_area['area_ha']).abs()
check_area['pct_diff'] = (check_area['abs_diff'] / check_area['area_ha']) * 100

area_mismatch = check_area[check_area['pct_diff'] > 5]
print(f"Number of farms with significant area mismatch (>5%): {len(area_mismatch)}")
print(area_mismatch[['farm_id','area_ha','calc_area_ha','abs_diff','pct_diff']].head(10))

Number of farms with significant area mismatch (>5%): 75
     farm_id  area_ha  calc_area_ha  abs_diff   pct_diff
6          7     0.11      0.116325  0.006325   5.750152
217      218     0.24      0.252111  0.012111   5.046115
242      243     0.15      0.159124  0.009124   6.082805
245      246     0.01      0.012899  0.002899  28.993789
356      357     0.21      0.220670  0.010670   5.080890
517      518     0.27      0.283822  0.013822   5.119134
553      554     0.03      0.027198  0.002802   9.341491
561      562     0.14      0.148265  0.008265   5.903897
649      650     0.12      0.126593  0.006593   5.494446
658      659     0.08      0.085566  0.005566   6.958068


#### Overlap

In [139]:
import geopandas as gpd

# Spatial join: find intersecting geometries
joined = gpd.sjoin(farm_boundaries, farm_boundaries, how='inner', predicate='intersects')

# Remove self-matches (same farm_id)
overlaps = joined[joined['farm_id_left'] != joined['farm_id_right']]

print("Number of overlapping boundaries:", len(overlaps))
print(overlaps[['farm_id_left','farm_id_right']].head())

overlap_pairs = overlaps[['farm_id_left','farm_id_right']].copy()

# Keep only unique pairs 
overlap_pairs = overlap_pairs[
    overlap_pairs['farm_id_left'] < overlap_pairs['farm_id_right']
]

print("Unique overlapping pairs:", len(overlap_pairs))
print(overlap_pairs)

Number of overlapping boundaries: 20
     farm_id_left  farm_id_right
2               3            757
7               8           1490
451           452            453
452           453            452
756           757              3
Unique overlapping pairs: 10
      farm_id_left  farm_id_right
2                3            757
7                8           1490
451            452            453
1898          1899           2370
1900          1901           2581
1920          1921           2581
2150          2151           2552
2159          2160           2568
2175          2176           2568
2423          2424           2436


In [142]:
# CSV file path
csv_file_path = os.path.join("C:/Users/Vasmai/dataset", "farm_boundaries_cleaned.csv")

# Save cleaned DataFrame directly to CSV
farm_cleaned.to_csv(csv_file_path, index=False) 

print(f"\nCleaned dataset exported to CSV: {csv_file_path}")


Cleaned dataset exported to CSV: C:/Users/Vasmai/dataset\farm_boundaries_cleaned.csv
